In [7]:
import os
os.environ["GOOGLE_API_KEY"]=os.getenv("GOOGLE_API_KEY")

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from uuid import uuid4

checkpointer = InMemorySaver()

agent = create_agent(
   model="google_genai:gemini-3.1-flash-lite",
   checkpointer= checkpointer,
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.1-flash-lite",
            trigger=("messages", 11),
            keep=("messages", 4),
        ),
    ],
)

config = {
    "configurable": {
        "thread_id": str(uuid4())
    }
}

In [ ]:
questions = [
    "My name is Avi.",
    "I study at Thapar Institute.",
    "I like Python.",
    "I am learning LangGraph.",
    "I want to build RAG applications.",
    "I want to be production level agentic ai engineer"
]

for q in questions:
    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": q
                }
            ]
        },
        config=config
    )
    length= len(result)
    print(result['messages'][-1].content)
    print(len(result["messages"]))



content=[{'type': 'text', 'text': 'It’s nice to meet you, Avi! How is your day going? Is there anything I can help you with today?', 'extras': {'signature': 'EjQKMgERTTIPqAkWsEpIj+Ho3UYcT7lq9mgKwVuX4te09ZR9RuqUXmpypTE+DCX4sExAWsHn'}}] additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019f793e-1aac-77c0-8991-f20755aeb3b9-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 6, 'output_tokens': 26, 'total_tokens': 32, 'input_token_details': {'cache_read': 0}}
2
content=[{'type': 'text', 'text': 'That’s great, Avi! Thapar Institute (TIET) in Patiala has a solid reputation, especially for engineering and technology. \n\nWhat are you currently studying there? And how are you finding the campus life so far?', 'extras': {'signature': 'EjQKMgERTTIParSrfz6t1hTrdOfEuv1Uu8w3Z/zpXE4KxzDRyUPSlVeO7PS1DOhcsNpZqw/v'}}] additional_kwargs={} response_metadata={'finis

In [6]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent=create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="google_genai:gemini-3.1-flash-lite",
            trigger=("tokens",550),
            keep=("tokens",200),
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token    

In [8]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~676 tokens, 8 messages
[HumanMessage(content='Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nThe user is conducting a comparative analysis of hotel accommodations across major international cities (Paris, London, Tokyo, New York City, and Dubai) based on a fixed three-tier pricing structure: Luxury ($350/night), Business ($180/night), and Budget ($75/night).\n\n## SUMMARY\n\n*   **Strategy:** A consistent tier-based framework (Luxury: $350, Business: $180, Budget: $75) has been applied to normalize comparisons across all target locations.\n*   **Completed Regions:** Research has been finalized for Paris, London, Tokyo, New York City, and Dubai.\n*   **Data Summary (Dubai):** \n    *   Luxury: Grand Hotel ($350/night, 5-star, spa, pool, gym).\n    *   Business: City Inn ($180/night, 4-star, business center).\n    *   Budget: Budget Stay ($75/night, 3-star, free Wi-Fi).\n*   *Note:* Similar data points were successfully retrieved and presented for the four

KeyboardInterrupt: 

In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import HumanMessage


def your_read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def your_send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"


In [8]:
agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[your_read_email_tool, your_send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "your_send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "your_read_email_tool": False,
            }
        ),
    ],
)


In [9]:
config = {"configurable":{"thread_id":"test123"}}
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)



In [10]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='15233fe6-0314-4b61-b77f-86c249668929'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'your_send_email_tool', 'arguments': '{"recipient": "john@test.com", "subject": "Hello", "body": "How are you?"}'}, '__gemini_function_call_thought_signatures__': {'9LPSL2bm': 'EjQKMgERTTIP5K57ZktC3iH5w1vfvM+3UiWVu065OGmUFA3Y5GAiH9z6QSXyyXAd5IKCgr4g'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f7950-2b1e-7900-8fb5-596c85e748a3-0', tool_calls=[{'name': 'your_send_email_tool', 'args': {'recipient': 'john@test.com', 'subject': 'Hello', 'body': 'How are you?'}, 'id': '9LPSL2bm', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 148, 'output_tokens': 39, 'total_tokens': 187, 'input_token_

In [11]:
from langgraph.types import Command
# Step 2: Approve
if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")
    
    result = agent.invoke(
        Command(
            resume={
                "decisions": [
                    {"type": "approve"}
                ]
            }
        ),
        config=config
    )
    
    print(f"✅ Result: {result['messages'][-1].content}")

⏸️ Paused! Approving...
✅ Result: [{'type': 'text', 'text': "OK. I've sent the email to john@test.com.", 'extras': {'signature': 'EjQKMgERTTIPt9FPCDcaYIFdd2pwVN8y2Xmsmr4LYtYZGyCXIGVyURd7crKS7wmzwhwISW8v'}}]


In [12]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='15233fe6-0314-4b61-b77f-86c249668929'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'your_send_email_tool', 'arguments': '{"recipient": "john@test.com", "subject": "Hello", "body": "How are you?"}'}, '__gemini_function_call_thought_signatures__': {'9LPSL2bm': 'EjQKMgERTTIP5K57ZktC3iH5w1vfvM+3UiWVu065OGmUFA3Y5GAiH9z6QSXyyXAd5IKCgr4g'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f7950-2b1e-7900-8fb5-596c85e748a3-0', tool_calls=[{'name': 'your_send_email_tool', 'args': {'recipient': 'john@test.com', 'subject': 'Hello', 'body': 'How are you?'}, 'id': '9LPSL2bm', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 148, 'output_tokens': 39, 'total_tokens': 187, 'input_token_

In [13]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver


def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email."""
    return f"Email sent to {recipient} with subject '{subject}'"

agent = create_agent(
    model="gpt-4o",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                },
                "read_email_tool": False,
            }
        ),
    ],
)

In [14]:
config = {"configurable": {"thread_id": "test-edit"}}

# Step 1: Request (with wrong info)
result = agent.invoke(
    {"messages": [HumanMessage(content="Send email to wrong@email.com with subject 'Test' and body 'Hello'")]},
    config=config
)

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
result